# 🥇 Laboratório do `GOLD_SQL` (`jobs/silver_to_gold.py`)

**O que este notebook é:** o `GOLD_SQL` desmontado peça por peça — cada CTE e cada função de janela rodando sozinha sobre um cenário mínimo e fabricado, para você *ver* o que cada construção faz antes de vê-las juntas.

**O que ele não é:** substituto de `tests/test_gold_sql.py`, que trava esses mesmos comportamentos como rede de regressão no CI.

**Cenário fabricado** (o mesmo dos testes): `m1` e `m2` disputam a categoria *varejo*; `m3` está sozinho em *serviços*; `m1` tem histórico no dia anterior (para o `LAG`); uma linha em USD não pode contar (filtro de moeda).

**Sumário**
1. Setup e views mínimas
2. O resultado completo
3. Desmontando: filtro de moeda e `FILTER (WHERE ...)`
4. Desmontando: broadcast join com a dimensão
5. Desmontando: `DENSE_RANK` e `LAG` (e o NULL do estreante)
6. Lendo o plano com `explain()`
7. Exercícios

> Convenção da pasta: roda de cima a baixo; `Kernel → Restart and Run All` se o estado embolar.

## 1. Setup e views mínimas

O `GOLD_SQL` lê duas views: `silver_events` (fatos) e `dim_merchants` (dimensão). Fabricamos as duas em memória — 9 linhas bastam para exercitar tudo.

In [ ]:
import sys

sys.path.insert(0, "../jobs")

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.master("local[2]")
    .appName("lab-gold")
    .config("spark.sql.shuffle.partitions", "2")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

from silver_to_gold import GOLD_SQL

TARGET, ANTERIOR = "2026-07-02", "2026-07-01"

silver = [
    # dt, merchant, customer, metodo, status, amount, moeda
    (ANTERIOR, "m1", "c01", "card", "approved", 100.0, "BRL"),  # historico p/ LAG
    (TARGET, "m1", "c02", "card", "approved", 100.0, "BRL"),
    (TARGET, "m1", "c03", "card", "approved", 100.0, "BRL"),
    (TARGET, "m1", "c04", "card", "approved", 100.0, "BRL"),
    (TARGET, "m1", "c05", "card", "approved", 999.0, "USD"),   # fora do filtro de moeda
    (TARGET, "m2", "c06", "card", "approved", 100.0, "BRL"),
    (TARGET, "m2", "c07", "card", "approved", 100.0, "BRL"),
    (TARGET, "m2", "c08", "card", "refunded", 50.0, "BRL"),    # estorno
    (TARGET, "m3", "c09", "pix", "approved", 100.0, "BRL"),    # 100% pix
]
spark.createDataFrame(
    silver,
    "dt STRING, merchant_id STRING, customer_id STRING, "
    "payment_method STRING, status STRING, amount DOUBLE, currency STRING",
).createOrReplaceTempView("silver_events")

spark.createDataFrame(
    [("m1", "Loja Um", "varejo", "SP"),
     ("m2", "Loja Dois", "varejo", "RJ"),
     ("m3", "Servico Tres", "servicos", "MG")],
    "merchant_id STRING, merchant_name STRING, category STRING, state STRING",
).createOrReplaceTempView("dim_merchants")

print("views prontas")

## 2. O resultado completo

O SQL inteiro, como o job roda (só trocando o `{target_dt}`). Guarde este resultado — as seções seguintes explicam de onde cada coluna saiu.

In [ ]:
gold = spark.sql(GOLD_SQL.format(target_dt=TARGET))
gold.orderBy("category", "rank_categoria").show(truncate=False)

Confira contra o cenário: `m1` tem `tx_total=3` (o USD ficou fora), `m2` tem estorno de 50 e taxa 2/3, `m3` tem `share_pix=1.0`, e só `m1` tem `gmv_dia_anterior` preenchido.

## 3. Desmontando: filtro de moeda e `FILTER (WHERE ...)`

A CTE `base` corta tudo que não é BRL **antes** de agregar — misturar moedas numa soma é erro clássico. E a CTE `agg` usa `FILTER (WHERE ...)`, a agregação condicional do SQL padrão — mais legível (e mais rápida) que `SUM(CASE WHEN ...)`:

In [ ]:
spark.sql(f'''
    SELECT
        merchant_id,
        COUNT(*)                                        AS tx_total,
        COUNT(*) FILTER (WHERE status = 'approved')     AS tx_aprovadas,
        SUM(amount) FILTER (WHERE status = 'approved')  AS gmv_aprovado,
        SUM(amount) FILTER (WHERE status = 'refunded')  AS valor_estornado
    FROM silver_events
    WHERE currency = 'BRL' AND dt = '{TARGET}'
    GROUP BY merchant_id
    ORDER BY merchant_id
''').show()

Compare `tx_total` com `tx_aprovadas` do `m2`: mesmo `GROUP BY`, recortes diferentes por linha — sem subquery, sem join consigo mesmo.

## 4. Desmontando: broadcast join com a dimensão

A dimensão tem 3 linhas (300 no dado real); os fatos, milhares. O hint `/*+ BROADCAST(m) */` manda a tabela pequena inteira para cada executor e **elimina o shuffle** do lado grande — a decisão comentada no próprio `GOLD_SQL`:

In [ ]:
enriquecido = spark.sql(f'''
    SELECT /*+ BROADCAST(m) */ e.merchant_id, e.amount, m.category, m.state
    FROM silver_events e
    LEFT JOIN dim_merchants m ON e.merchant_id = m.merchant_id
    WHERE e.dt = '{TARGET}' AND e.currency = 'BRL'
''')
enriquecido.show()
enriquecido.explain()  # procure "BroadcastHashJoin" no plano

## 5. Desmontando: `DENSE_RANK` e `LAG` (e o NULL do estreante)

As duas janelas do `GOLD_SQL`, isoladas:

- `DENSE_RANK() OVER (PARTITION BY dt, category ORDER BY gmv DESC)` — ranking **dentro da categoria** no dia: `m1` e `m2` disputam varejo; `m3`, sozinho em serviços, também é rank 1.
- `LAG(gmv) OVER (PARTITION BY merchant_id ORDER BY dt)` — o valor do **mesmo merchant** no dia anterior. `m2` estreou hoje: dia anterior e variação ficam **NULL** — e NULL é o comportamento certo, não zero (zero mentiria "estável").

In [ ]:
spark.sql('''
    WITH gmv_dia AS (
        SELECT dt, merchant_id,
               SUM(amount) FILTER (WHERE status = 'approved') AS gmv
        FROM silver_events
        WHERE currency = 'BRL'
        GROUP BY dt, merchant_id
    )
    SELECT dt, merchant_id, gmv,
           LAG(gmv) OVER (PARTITION BY merchant_id ORDER BY dt) AS gmv_dia_anterior
    FROM gmv_dia
    ORDER BY merchant_id, dt
''').show()

Veja o `m1`: a linha do dia 02 enxerga o gmv do dia 01. E o `m2` no dia 02: `gmv_dia_anterior` NULL — estreante.

Troque o `LAG` por `LAG(gmv, 1, 0)` (default 0) e observe a variação do `m2` virar um número que **mente**: +∞% ou divisão por zero disfarçada. É por isso que o job propaga o NULL.

## 6. Lendo o plano com `explain()`

O hábito que separa "rodou" de "sei o que rodou": procure no plano o `BroadcastHashJoin` (seção 4), os `HashAggregate` das agregações e o `Window` das janelas — e note que janela exige `Sort` dentro da partição:

In [ ]:
gold.explain()

## 7. Exercícios

1. **Rank vs row:** troque `DENSE_RANK` por `ROW_NUMBER` e por `RANK` na consulta da seção 5 (adaptando-a). Crie um empate de gmv e explique a diferença entre os três no resultado.
2. **Métrica nova:** adicione `share_por_canal` (aprovado por `payment_method` ÷ total) na consulta da seção 3, usando só `FILTER (WHERE ...)`.
3. **Quebre a ponte de propósito:** renomeie mentalmente `amount` para `valor` no silver — qual pedaço do `GOLD_SQL` quebraria, e qual teste do repo pegaria isso antes da produção? (Resposta em `tests/test_ponte_silver_gold.py`.)
4. **Feche o ciclo:** o comportamento mais interessante que você viu aqui já está protegido em `tests/test_gold_sql.py`? Se não, escreva o teste.

In [ ]:
spark.stop()